# GPU IVF vector search: full benchmark run

Runs on **Kaggle** (GPU T4/P100, Internet on) or **Google Colab** (GPU runtime).

* Kaggle with real data: *Add data* → `ltcmdrdata/plain-text-wikipedia-202011`, keep `SOURCE = "kaggle_wikipedia"`.
* Colab (or Kaggle without the dataset): `SOURCE = "hf_wikipedia"` streams Wikipedia from Hugging Face.
* Scaling beyond the real data: a second run with `SOURCE = "synthetic"` (up to 5M vectors on a 16 GB GPU).

Everything ends up in `results/<dataset>/` (CSVs, `plots/`, `summary.md`); the last cell zips it.

In [ ]:
!nvidia-smi
!nvcc --version | tail -2

In [ ]:
import os
REPO = "https://github.com/navyasg0609/GPU_Accelerated_Vector_Indexing.git"
WORK = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
os.chdir(WORK)
if not os.path.exists("GPU_Accelerated_Vector_Indexing"):
    !git clone -q {REPO}
os.chdir("GPU_Accelerated_Vector_Indexing")
!git log --oneline -1

In [ ]:
# FAISS with GPU support; falls back to the CPU build if no matching wheel exists.
!pip install -q faiss-gpu-cu12 || pip install -q faiss-cpu
!pip install -q sentence-transformers scikit-learn datasets
import faiss; print("faiss", faiss.__version__, "GPUs:", faiss.get_num_gpus())

In [ ]:
!bash scripts/build.sh 2>&1 | tail -5
!./build/test_correctness

In [ ]:
# ---- configuration (all optional; defaults shown in scripts/run_all.sh) ----
SOURCE = "kaggle_wikipedia"          # kaggle_wikipedia | hf_wikipedia | synthetic
SCALES = "100000 500000 1000000"     # e.g. "100000 1000000 5000000" for synthetic
RUNS = 5
os.environ.update(SOURCE=SOURCE, SCALES=SCALES, RUNS=str(RUNS))
!bash scripts/run_all.sh

In [ ]:
from IPython.display import Markdown, Image, display
import glob
res = sorted(glob.glob("results/*/summary.md"))[-1]
display(Markdown(open(res).read()))
for p in sorted(glob.glob(os.path.join(os.path.dirname(res), "plots", "*.png"))):
    display(Image(p))

In [ ]:
!zip -qr results.zip results && ls -lh results.zip
# Download results.zip (Kaggle: Output tab; Colab: files panel) and commit it into results/.